In [ ]:
import cv2
import torch
import numpy as np

def init_model(ckpt_path):
    import sys
    sys.path.insert(0, "/home/vuthede/OMS_AI/.github/edgeai-yolox")
    from yolox.models import YOLOXOMS, YOLOPAFPN, YOLOXHeadKPTS, YOLOXHead as YOLOXObjectHead
    human_params = {
        "num_classes": 1,
        "num_kpts": 17,
        "default_sigmas": False,
        "data_dir": "/home/vuthede/fiftyone/coco-2017/validation",
        "train_ann": "person_keypoints_val2017_with_random_bimometry.json",  
        "name": "val2017",
        "flip_prob": 0.5,
        "hsv_prob": 0.5,
        "degrees": 10.0,
        "translate": 0.1,
        "mosaic_scale": [0.8, 1.2],
        "mixup_scale": [0.5, 1.5],
        "shear": 2.0,
        "enable_mixup": True,
        "mosaic_prob": 1.0,
        "mixup_prob": 1.0,
    }   
    face_params = {
        "num_classes": 1,
        "num_kpts": 5,
        "default_sigmas": False,
        "data_dir": "/media/vuthede/Lexar/data/face_detection/widerface/train",
        "train_ann": "annotations_coco.json",  
        "name": "images",
        "flip_prob": 0.5,
        "hsv_prob": 0.5,
        "degrees": 10.0,
        "translate": 0.1,
        "mosaic_scale": [0.8, 1.2],
        "mixup_scale": [0.5, 1.5],
        "shear": 2.0,
        "enable_mixup": True,
        "mosaic_prob": 1.0,
        "mixup_prob": 1.0,
    }       
    object_params = {
        "num_classes": 4,  # phone, ciga, food, beverage
        "data_dir": "/media/vuthede/Lexar/phone_food_smoking_sampling_full",
        "train_ann": "sampling_20k.json",
        "name": "images",
        "flip_prob": 0.5,
        "hsv_prob": 0.5,
        "degrees": 10.0,
        "translate": 0.1,
        "mosaic_scale": [0.8, 1.2],
        "mixup_scale": [0.5, 1.5],
        "shear": 2.0,
        "enable_mixup": True,
        "mosaic_prob": 1.0,
        "mixup_prob": 1.0,
    }
    depth = 0.33
    width = 0.50
    in_channels = [256, 512, 1024]
    act = "relu"
    backbone = YOLOPAFPN(depth, width, act=act,in_channels=in_channels, conv_focus=True)
    head_human = YOLOXHeadKPTS(human_params["num_classes"], width, in_channels=in_channels, act=act, num_kpts=human_params["num_kpts"], default_sigmas=human_params["default_sigmas"])
    head_face = YOLOXHeadKPTS(face_params["num_classes"], width, in_channels=in_channels, act=act, num_kpts=face_params["num_kpts"], default_sigmas=face_params["default_sigmas"])
    head_object = YOLOXObjectHead(object_params["num_classes"], width=width, in_channels=in_channels)
    model = YOLOXOMS(backbone, head_dict={"human": head_human, "face": head_face, "object": head_object})
    print("Init model!!!")
    
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt["model"])
    print(f"Load checkpoint from {ckpt_path}")
    
    return model

def preprocess_frame(frame, target_size=(640, 640)):
    """Preprocess the frame: resize with padding, normalize, and convert to tensor."""
    # Convert BGR (OpenCV) to RGB
    img = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Get original dimensions
    h, w = img.shape[:2]
    target_h, target_w = target_size
    
    # Calculate scaling factor to maintain aspect ratio
    scale = min(target_h / h, target_w / w)
    new_h, new_w = int(h * scale), int(w * scale)
    
    # Resize image
    resized_img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
    
    # Create a blank canvas (black padding)
    padded_img = np.zeros((target_h, target_w, 3), dtype=np.uint8)
    
    # Place resized image in the center
    top = (target_h - new_h) // 2
    left = (target_w - new_w) // 2
    padded_img[top:top + new_h, left:left + new_w] = resized_img
    
    # Normalize to [0, 1] and apply YOLOX standard mean/std
    img_tensor = torch.from_numpy(padded_img).float() / 255.0
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3)
    img_tensor = (img_tensor - mean) / std
    
    # Permute to (C, H, W) and add batch dimension
    img_tensor = img_tensor.permute(2, 0, 1).unsqueeze(0)
    
    return img_tensor, (scale, top, left)

In [ ]:
frame = cv2.imread("/home/vuthede/fiftyone/coco-2017/validation/val2017/000000000785.jpg") # 1 person
# frame = cv2.imread("/home/vuthede/fiftyone/coco-2017/validation/val2017/000000008690.jpg") # 2 people
# frame = cv2.imread("/home/vuthede/fiftyone/coco-2017/validation/val2017/000000001000.jpg") # many people
# frame = cv2.resize(frame, (640, 640))

frame = cv2.flip(frame, 1)
model = init_model("/home/vuthede/OMS_AI/.github/edgeai-yolox/YOLOX_outputs/train_s_oms/latest_ckpt.pth")
model.eval()
img_tensor, (scale, top, left) = preprocess_frame(frame)
print(img_tensor.shape, scale, top, left)


## Human

In [ ]:
result = model(img_tensor)
human = result["human"]
print(f'Human shape: {human.shape}')
boxes = human[..., :4]  # [B, 8400, 4]
obj_score = human[..., 4:5]  # [B, 8400, 1]
cls_score = human[..., 5:5 + 1]  # [B, 8400, num_classes]
scores = obj_score * cls_score.max(dim=-1, keepdim=True)[0] 
mask = scores.squeeze(-1) > 0.5

kpts = human[..., 6:6+3*17] 
print(f'Kpts shape: {kpts.shape}')


print(f'Boxes shape: {boxes.shape}. Obj score shape: {obj_score.shape}. Cls score shape: {cls_score.shape}')


import matplotlib.pyplot as plt
filtered_boxes = boxes[mask]
filtered_kpts = kpts[mask]


print(f'Filtered boxes shape: {filtered_boxes.shape}. Filter kpts shape: {filtered_kpts.shape}')
box_result = filtered_boxes.clone()
box_result[:, 0] -= filtered_boxes[:, 2] / 2  # x1 = center_x - w/2
box_result[:, 1] -= filtered_boxes[:, 3] / 2  # y1 = center_y - h/2
box_result[:, 2] = filtered_boxes[:, 0] + filtered_boxes[:, 2]/2      # x2 = x1 + w
box_result[:, 3] = filtered_boxes[:, 1] + filtered_boxes[:, 3]/2    # y2 = y1 + h
box_result = box_result.detach().cpu().numpy()

# Using scale, top, left to convert the box back to original size
box_result[:, :4] /= scale
box_result[:, 0] -= left
box_result[:, 1] -= top
box_result[:, 2] -= left
box_result[:, 3] -= top

# Using scale, top, left to convert the kpts back to original size
# kpts: [1, 17*3]
# filtered_kpts = filtered_kpts.squeeze(0)
filtered_kpts[:, 0::3] -= left  # x coordinates
filtered_kpts[:, 1::3] -= top   # y coordinates
filtered_kpts[:, 0::3] /= scale
filtered_kpts[:, 1::3] /= scale


for box in box_result:
    x1, y1, x2, y2 = box
    x1, y1, x2, y2 = int(x1), int(y1), int(x2), int(y2)
    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
    
    
# Plot keypoints
for kpts in filtered_kpts:
    for i in range(0, len(kpts), 3):
        x, y, v = kpts[i:i+3]
        x, y = int(x), int(y)
        # if v > 0:
        cv2.circle(frame, (x, y), 3, (255, 0, 0), -1)   
    
plt.imshow(frame)


In [ ]:
filtered_kpts

In [ ]:

print(mask.shape)